# 04. 기준 모델 + 공통 지표 (10/30) — 김율이 C1·C2

| 셀 | 내용 |
|---|---|
| 0 | 설정 (Drive, 레포 pull, config) |
| 1 | 기준 모델 4개 fit → valid·test 예측값을 `predictions/`에 저장 |
| 2 | 평점 지표: RMSE·MAE × 범위(all/seen) × 구간(cold/mid/active) |
| 3 | 랭킹 지표: 프로토콜 ①②③ HR@10·NDCG@10 (전체 사용자 / p3_users 공통 표본) |
| 4 | 성공 기준 판정 함수 동작 확인 (①, ③) |

- 규칙: `docs/EVAL_RULES.md` (신뢰구간은 평가 담당이 한꺼번에 / 정수 번호 배열 / 판정 규칙)
- 다른 모델(MF, CBF, 하이브리드)은 `predictions/`에 예측값을 올리거나 `score`·`score_rows`를 가진 객체를 넘기면 같은 함수로 채점된다


In [ ]:
# ===== 0. 설정 =====
import os, sys, subprocess, time
try:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO = '/content/team10-recsys'
    if not os.path.exists(REPO):
        subprocess.run(['git', 'clone', '-q', 'https://github.com/qhfl3644/team10-recsys.git', REPO], check=True)
    else:
        subprocess.run(['git', '-C', REPO, 'pull', '-q'], check=True)
except ImportError:
    REPO = os.environ.get('TEAM10_REPO', '..')
sys.path.insert(0, REPO)
from config import *

import numpy as np
import pandas as pd
from src.models.baselines import BASELINES
from src.eval import metrics as M

OUT = f'{RESULT_DIR}/baselines'
os.makedirs(OUT, exist_ok=True)


## 1. 기준 모델 fit → 예측값 저장

전역 평균·사용자 평균·상품 평균(평점 트랙 기준 모델 3개) + 인기도(Top-N 기준 모델). `predictions/{ds}/{모델}_{split}.parquet`에 `user_id, item_id, uidx, iidx, pred`로 저장된다.

In [ ]:
# ===== 1. 기준 모델 =====
data, models, preds = {}, {}, {}
for ds in DATASETS:
    t0 = time.time()
    data[ds] = M.load_data(ds)
    d = data[ds]
    models[ds] = {name: cls().fit(d.train, d.n_users, d.n_items) for name, cls in BASELINES.items()}
    preds[ds] = {}
    for split in ['valid', 'test']:
        df = getattr(d, split)
        preds[ds][split] = {name: m.predict(df['uidx'].values, df['iidx'].values) for name, m in models[ds].items()}
        for name, p in preds[ds][split].items():
            M.save_predictions(ds, name, split, df, p)
    print(f'{ds}: users={d.n_users:,} items={d.n_items:,} train={len(d.train):,} → 예측값 8개 저장 ({time.time()-t0:.0f}s)')


## 2. 평점 지표 (test)

`scope=seen`은 정답 상품이 train에 있는 평점만(순수 MF 비교용). 구간은 train 평점 수 3~4 / 5~9 / 10+.

In [ ]:
# ===== 2. 평점 지표 =====
rating_tables = []
for ds in DATASETS:
    rt = M.rating_table(preds[ds]['test'], data[ds].test, data[ds]); rt.insert(0, 'dataset', ds)
    rating_tables.append(rt)
rating_all = pd.concat(rating_tables, ignore_index=True)
rating_all.to_csv(f'{OUT}/rating_test.csv', index=False)
rating_all[(rating_all.scope == 'all')].pivot_table(index=['dataset', 'model'], columns='group', values='rmse').round(4)


## 3. 랭킹 지표 (test)

①②는 전체 사용자와 `p3_users`(1만 명) 두 가지, ③은 `p3_users`. **프로토콜 간 비교와 성공 기준 ③은 p3_users 공통 표본으로.** 점수가 사용자별 상수인 모델(전역/사용자 평균)은 ①②에서 ≈0.10이 나와야 정상(동점 무작위 처리 확인).

In [ ]:
# ===== 3. 랭킹 지표 =====
rank_tables, hits = [], {}
for ds in DATASETS:
    t0 = time.time()
    p3 = M.load_p3_users(ds)
    a, _ = M.ranking_eval(models[ds], data[ds], 'test', protocols=('p1', 'p2')); a['users'] = 'all'
    b, hits[ds] = M.ranking_eval(models[ds], data[ds], 'test', protocols=('p1', 'p2', 'p3'), users=p3); b['users'] = 'p3_users'
    rk = pd.concat([a, b], ignore_index=True); rk.insert(0, 'dataset', ds); rank_tables.append(rk)
    print(f'{ds}: 랭킹 평가 완료 ({time.time()-t0:.0f}s)')
rank_all = pd.concat(rank_tables, ignore_index=True)
rank_all.to_csv(f'{OUT}/ranking_test.csv', index=False)
rank_all[rank_all.users == 'p3_users'].pivot_table(index=['dataset', 'model'], columns='protocol', values='HR10').round(4)


## 4. 성공 기준 판정 함수 확인

아직 MF·CBF·하이브리드가 없으므로 기준 모델끼리 넣어 함수가 도는지만 본다. 기준 ③ 예시(사용자 평균 vs 인기도)는 당연히 `passed=False`가 나와야 한다.

In [ ]:
# ===== 4. 판정 함수 =====
for ds in DATASETS:
    p = preds[ds]['test']
    c1 = M.criterion_rating_vs_baselines(p['item_mean'], {k: p[k] for k in ['global_mean', 'user_mean', 'item_mean']}, data[ds].test, data[ds])
    c3 = M.criterion_topn_vs_popularity(hits[ds], hybrid='user_mean', popularity='popularity', protocol='p3')
    print(f'=== {ds}'); print(c1.round(4).to_string(index=False)); print(c3)
    M.save_json({'criterion1_example': c1.to_dict('records'), 'criterion3_example': c3}, f'{OUT}/{ds}_criteria_example.json')


## 다음

- MF(혜림님)·CBF(지훈님) 예측값이 `predictions/`에 올라오면 2번 셀의 `preds[ds]['test']`에 `M.load_predictions(ds, 'mf', 'test', data[ds].test)`로 추가해서 같은 표를 다시 만든다.
- Top-N 트랙(지훈님)은 `score(uidx, iidx)`·`score_rows(uidx)`를 가진 객체를 3번 셀 `models[ds]`에 추가한다.
- 기준 ② `M.criterion_cold_hybrid`, 기준 ④는 Toys에서 `criterion_rating_vs_baselines`. 규칙은 `docs/EVAL_RULES.md`.
